# AWS Agent Registry를 사용하여 Runtime에 도구와 에이전트 검색

## 개요

이 Notebook에서는 에이전트가 하드코딩된 통합 없이 runtime에 다른 에이전트와 MCP 서버를 검색할 수 있게 하는 semantic search catalog인 **AWS Agent Registry**를 보여 줍니다.

Orchestrator 에이전트는 다음 작업을 수행합니다.
1. 자연어로 Registry를 **검색**하여 관련 MCP 서버와 A2A 에이전트 탐색
2. 실제 연결 **인스턴스화**: Amazon Bedrock AgentCore Gateway MCP 서버에는 `MCPClient`, A2A 에이전트에는 `@tool` wrapper 사용
3. 동적으로 검색된 기능만 사용하여 사용자 요청 **실행**

### AWS Agent Registry를 사용하는 이유

대부분의 에이전트 시스템에서는 통합이 하드코딩되어 있어 에이전트가 build 시점에 호출할 API를 정확히 알고 있습니다. 다음과 같은 경우 문제가 발생합니다.
- 여러 팀이 새 기능을 독립적으로 게시하는 경우(에이전트에서 사용하려면 재배포 필요)
- 모든 통합을 미리 알지 못한 채 하나의 orchestrator를 여러 도메인에서 사용하려는 경우
- Endpoint URL 또는 인증 요구 사항이 변경되는 경우(모든 consumer가 업데이트 필요)

**AWS Agent Registry**는 **Registry 기반 검색**으로 이 문제를 해결합니다. MCP 서버와 에이전트가 상세한 설명과 함께 catalog에 자신을 등록하고, runtime에 orchestrator가 자연어로 catalog를 검색하여 필요한 항목을 찾습니다. 새 기능은 재배포 없이 즉시 사용할 수 있습니다.

아래 다이어그램은 두 방식을 나란히 보여 줍니다. **Registry를 사용하지 않으면**(왼쪽) endpoint가 하드코딩되어 새 에이전트를 추가할 때 코드 변경과 재배포가 필요합니다. **Registry를 사용하면**(오른쪽) orchestrator가 semantic search로 기능을 검색하고 새 통합을 등록 즉시 사용할 수 있습니다.

<img src="./images/With_Vs_Without_AWS_Agent_Registry.png" alt="AWS Agent Registry 사용 여부 비교" width="900"/>

### 사용 사례: 주문 관리 및 고객 서비스

Orchestrator 에이전트는 다음 항목을 동적으로 검색하여 고객의 주문 관련 작업을 지원합니다.
- 주문 데이터 조회(상태 확인, 주문 업데이트)를 위한 **MCP 서버**
- 비즈니스 로직 추론(가격/할인, 반품/환불)을 위한 **A2A 에이전트**

### 솔루션 개요

<img src="./images/OrderManagement_AWS_Agent_Registry_Flow.png" alt="주문 관리 AWS Agent Registry 흐름" width="900"/>

### 지원 서비스
- **Amazon Bedrock AgentCore Gateway** — AWS Lambda 함수 앞에 위치하는 관리형 MCP 서버(OAuth2 인바운드 인증)
- **Amazon Bedrock AgentCore Runtime** — IAM(SigV4) 인증으로 A2A 에이전트 호스팅
- **Amazon Bedrock** — Orchestrator 에이전트용 foundation model(Claude Sonnet 4.6)
- **AWS Lambda** — MCP 서버 구현용 backend
- **Amazon Cognito** — Gateway 인증용 OAuth2 token provider

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10 이상
* Jupyter Notebook(Python kernel)
* Amazon Bedrock, Amazon Bedrock AgentCore, AWS Lambda, IAM, Amazon Cognito, Amazon ECR 및 AWS CodeBuild 권한이 있는 AWS 자격 증명
* Claude Sonnet 4에 대한 Amazon Bedrock 모델 액세스 활성화
* boto3 >= 1.42.87

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1단계: 인프라 배포 - MCP Server(Gateway) + A2A Agent(Runtime)
# ═══════════════════════════════════════════════════════════════════════════════

# ─── 1.0 종속성 설치 ────────────────────────────────────────────────────────
import subprocess
import sys
import os

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        "-q",
        "strands-agents>=0.1.0",
        "strands-agents[a2a]>=0.1.0",
        "boto3>=1.42.87",
        "bedrock-agentcore>=1.0.0",
        "bedrock-agentcore-starter-toolkit>=0.1.24",
        "mcp>=1.0.0",
        "requests>=2.31.0",
    ]
)

# boto3 버전에 AgentCore Registry 기본 지원이 포함되는지 확인
import boto3 as _b3

assert tuple(int(x) for x in _b3.__version__.split(".")) >= (1, 42, 87), (
    f"boto3 >= 1.42.87 required for native Registry support, got {_b3.__version__}"
)
print(f"boto3 {_b3.__version__} — native AgentCore Registry support ✓")

# ═══════════════════════════════════════════════════════════════════════════════
# 이 셀은 orchestrator가 runtime에 검색할 수 있도록 Registry(2단계)에 등록할
# 모든 MCP 서버와 에이전트를 배포합니다.
#
#   MCP Server(AgentCore Gateway 사용):
#     • get_order_status — 주문 세부 정보, 품목, 추적 및 배송 조회
#     • update_order     — 주문 취소 또는 배송 주소 변경
#     Backend: Lambda 함수, 인증: Cognito OAuth2 → JWT
#
#   A2A Agent(AgentCore Runtime 사용):
#     • Pricing Agent          — 할인 등급, promo code, 가격 기록
#     • Customer Support Agent — 반품, 환불, escalation policy
#     Backend: Runtime의 Docker container, 인증: IAM SigV4
#
# 이 셀이 AWS 리소스 생성에 집중하도록 helper 코드(Lambda source, agent source,
# zip packaging)는 utils.py에 있습니다.
# ═══════════════════════════════════════════════════════════════════════════════

# ─── Import + AWS 클라이언트 ─────────────────────────────────────────────────
import boto3
import json
import time
from datetime import datetime
from utils import (
    ORDER_MANAGEMENT_LAMBDA_CODE,
    ORDER_TOOL_SCHEMAS,
    make_lambda_zip,
    write_agent_files,
)
from bedrock_agentcore_starter_toolkit import Runtime

session = boto3.Session()
region = session.region_name or "us-west-2"
os.environ["AWS_DEFAULT_REGION"] = region

sts_client = session.client("sts")
account_id = sts_client.get_caller_identity()["Account"]
iam_client = session.client("iam")
lambda_client = session.client("lambda")
cognito_client = session.client("cognito-idp")
sm_client = session.client("secretsmanager")

# Control plane - Gateway, Runtime 및 Registry CP 작업에 사용
cp_client = session.client("bedrock-agentcore-control")
# Data plane - Runtime 호출에 사용
agentcore_client = session.client("bedrock-agentcore")
# Registry data plane - SearchRegistryRecords에 사용(Registry와 동일한 리전)
dp_client = session.client("bedrock-agentcore-registry")

timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
print(f"Account: {account_id} | Region: {region} | Timestamp: {timestamp}")

# ─── 1a. Lambda - 주문 관리 backend ──────────────────────────────────────────
lambda_role_name = f"LambdaMCPRole-{timestamp}"
lambda_role_resp = iam_client.create_role(
    RoleName=lambda_role_name,
    AssumeRolePolicyDocument=json.dumps(
        {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Effect": "Allow",
                    "Principal": {"Service": "lambda.amazonaws.com"},
                    "Action": "sts:AssumeRole",
                }
            ],
        }
    ),
)
lambda_role_arn = lambda_role_resp["Role"]["Arn"]
iam_client.attach_role_policy(
    RoleName=lambda_role_name,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)
time.sleep(10)

lambda_arns = {}
fname = f"order-management-mcp-{timestamp}"
resp = lambda_client.create_function(
    FunctionName=fname,
    Runtime="python3.13",
    Role=lambda_role_arn,
    Handler="lambda_function.lambda_handler",
    Code={"ZipFile": make_lambda_zip(ORDER_MANAGEMENT_LAMBDA_CODE)},
    Timeout=30,
)
lambda_arns["order-management-mcp"] = resp["FunctionArn"]
print(f"✓ Lambda: {fname}")

# ─── 1b. Cognito - Gateway 인증용 OAuth2 token provider ──────────────────────
pool_resp = cognito_client.create_user_pool(
    PoolName=f"gateway-pool-{timestamp}",
    Policies={
        "PasswordPolicy": {
            "MinimumLength": 8,
            "RequireUppercase": False,
            "RequireLowercase": False,
            "RequireNumbers": False,
            "RequireSymbols": False,
        }
    },
)
user_pool_id = pool_resp["UserPool"]["Id"]

resource_server_id = f"gateway-api-{timestamp}"
cognito_client.create_resource_server(
    UserPoolId=user_pool_id,
    Identifier=resource_server_id,
    Name=f"Gateway API {timestamp}",
    Scopes=[
        {"ScopeName": "read", "ScopeDescription": "Read"},
        {"ScopeName": "write", "ScopeDescription": "Write"},
    ],
)

app_client_resp = cognito_client.create_user_pool_client(
    UserPoolId=user_pool_id,
    ClientName=f"gateway-client-{timestamp}",
    GenerateSecret=True,
    AllowedOAuthFlows=["client_credentials"],
    AllowedOAuthFlowsUserPoolClient=True,
    AllowedOAuthScopes=[f"{resource_server_id}/read", f"{resource_server_id}/write"],
)
client_id = app_client_resp["UserPoolClient"]["ClientId"]

# client_secret은 환경 변수나 Notebook memory가 아닌 Secrets Manager에 저장
secret_name = f"gateway-client-secret-{timestamp}"
_client_secret = cognito_client.describe_user_pool_client(UserPoolId=user_pool_id, ClientId=client_id)[
    "UserPoolClient"
]["ClientSecret"]
sm_client.create_secret(Name=secret_name, SecretString=_client_secret)
del _client_secret  # Notebook memory에서 즉시 제거
print(f"✓ Client secret stored in Secrets Manager: {secret_name}")

domain_name = f"gateway-{timestamp}"
cognito_client.create_user_pool_domain(Domain=domain_name, UserPoolId=user_pool_id)
cognito_domain = f"{domain_name}.auth.{region}.amazoncognito.com"
print(f"✓ Cognito: pool={user_pool_id}")

scopes = f"{resource_server_id}/read {resource_server_id}/write"

# ─── 1c. AgentCore Gateway - Lambda 앞에 위치하는 관리형 MCP 서버 ────────────
gateway_role_name = f"AgentCoreGatewayRole-{timestamp}"
iam_client.create_role(
    RoleName=gateway_role_name,
    AssumeRolePolicyDocument=json.dumps(
        {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Effect": "Allow",
                    "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                    "Action": "sts:AssumeRole",
                }
            ],
        }
    ),
)
gateway_role_arn = f"arn:aws:iam::{account_id}:role/{gateway_role_name}"
iam_client.put_role_policy(
    RoleName=gateway_role_name,
    PolicyName="LambdaInvoke",
    PolicyDocument=json.dumps(
        {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Effect": "Allow",
                    "Action": "lambda:InvokeFunction",
                    "Resource": f"arn:aws:lambda:{region}:{account_id}:function:*",
                }
            ],
        }
    ),
)
time.sleep(10)

gateway_resp = cp_client.create_gateway(
    name=f"demo-gateway-{timestamp}",
    roleArn=gateway_role_arn,
    protocolType="MCP",
    protocolConfiguration={"mcp": {"supportedVersions": ["2025-03-26", "2025-06-18"]}},
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration",
            "allowedClients": [client_id],
        }
    },
)
gateway_id = gateway_resp["gatewayId"]
print(f"  Gateway creating: {gateway_id} ...")
while True:
    status = cp_client.get_gateway(gatewayIdentifier=gateway_id)
    if status.get("status") == "READY":
        gateway_url = status.get("gatewayUrl")
        break
    time.sleep(5)
print(f"✓ Gateway ready: {gateway_url}")

target_ids = {}
resp = cp_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=f"order-management-target-{timestamp}",
    targetConfiguration={
        "mcp": {
            "lambda": {
                "lambdaArn": lambda_arns["order-management-mcp"],
                "toolSchema": {"inlinePayload": ORDER_TOOL_SCHEMAS},
            }
        }
    },
    credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
)
tid = resp["targetId"]
target_ids["order-management-target"] = tid
while True:
    if cp_client.get_gateway_target(gatewayIdentifier=gateway_id, targetId=tid).get("status") == "READY":
        break
    time.sleep(10)
print(f"✓ Gateway target ready: {tid}")

# ─── 1d. A2A Agent - AgentCore Runtime에 배포 ────────────────────────────────
write_agent_files()

# 삭제된 에이전트 업데이트 오류를 방지하도록 오래된 starter toolkit 구성 제거
if os.path.exists(".bedrock_agentcore.yaml"):
    os.remove(".bedrock_agentcore.yaml")

pricing_rt = Runtime()
pricing_rt.configure(
    entrypoint="pricing_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="a2a_requirements.txt",
    region=region,
    agent_name="pricing_agent",
    protocol="A2A",
)
pricing_launch = pricing_rt.launch(auto_update_on_conflict=True)
pricing_agent_id = pricing_launch.agent_id
print(f"✓ Pricing Agent deployed: {pricing_agent_id}")

support_rt = Runtime()
support_rt.configure(
    entrypoint="customer_support_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="a2a_requirements.txt",
    region=region,
    agent_name="customer_support_agent",
    protocol="A2A",
)
support_launch = support_rt.launch(auto_update_on_conflict=True)
support_agent_id = support_launch.agent_id
print(f"✓ Customer Support Agent deployed: {support_agent_id}")

agent_arns = {}
for name, aid in [
    ("pricing_agent", pricing_agent_id),
    ("customer_support_agent", support_agent_id),
]:
    resp = cp_client.get_agent_runtime(agentRuntimeId=aid)
    agent_arns[name] = resp["agentRuntimeArn"]

# ─── Registry metadata - 2단계에서 등록할 항목 설명 ──────────────────────────
registry_records = {
    "order_management_mcp": {
        "protocol": "MCP",
        "description": "Order data tools - get order status, tracking, items, shipping details, cancel or change address",
        "tools": ORDER_TOOL_SCHEMAS,
    },
    "pricing_agent": {
        "protocol": "A2A",
        "description": "Pricing only - discount tiers, promo codes, price history. Never handles returns or refunds",
    },
    "customer_support_agent": {
        "protocol": "A2A",
        "description": "Returns and refunds only - return eligibility, refund calculation, complaints, escalations",
    },
}

# ─── 요약 - 배포된 모든 리소스 ARN ──────────────────────────────────────────
print("\n" + "═" * 70)
print("DEPLOYED RESOURCES")
print("═" * 70)
print(f"  Lambda ARN:            {lambda_arns['order-management-mcp']}")
print(f"  Gateway URL:           {gateway_url}")
print(f"  Gateway Target ID:     {tid}")
print(f"  Cognito User Pool:     {user_pool_id}")
print(f"  Secret Name:           {secret_name}")
print(f"  Pricing Agent ARN:     {agent_arns['pricing_agent']}")
print(f"  Support Agent ARN:     {agent_arns['customer_support_agent']}")
print("═" * 70)

---

## 2단계: AWS Agent Registry 생성 및 모든 레코드 등록

이 단계는 데모의 핵심입니다. Registry를 생성하고 MCP 서버와 A2A 에이전트를 검색 가능한 레코드로 등록한 후 승인합니다. 그런 다음 semantic search가 자연어 query에 적합한 기능을 찾는지 확인합니다.

### 2a단계: Registry 생성

In [ ]:
# MCP 도구와 A2A 에이전트를 등록할 검색 가능한 catalog인 새 Registry를 생성합니다.
# Orchestrator는 runtime에 이 catalog를 query하여 사용 가능한 기능을 검색합니다.
#
# autoApproval=False이면 레코드가 검색 결과에 표시되기 전에 명시적으로 승인해야 하므로
# 거버넌스 검토 워크플로를 사용할 수 있습니다.
reg = cp_client.create_registry(
    name="OrderManagementRegistry",
    description="Registry for Order Management & Customer Service - agent discovers tools and agents via semantic search",
    approvalConfiguration={"autoApproval": False},
)
REGISTRY_ARN = reg["registryArn"]
REGISTRY_ID = REGISTRY_ARN.split("/")[-1]
print(f"Registry: {REGISTRY_ID}")
print(f"ARN: {REGISTRY_ARN}")

# 레코드를 생성하기 전에 Registry가 READY 상태가 될 때까지 대기
while True:
    r = cp_client.get_registry(registryId=REGISTRY_ID)
    if r["status"] == "READY":
        print("Registry status: READY")
        break
    print(f"Registry status: {r['status']} - waiting...")
    time.sleep(5)

> ✅ **확인 지점**: Registry ID(UUID 형식)와 ARN이 표시되어야 합니다. Registry는 `autoApproval: False`로 생성되므로 레코드가 검색 결과에 표시되기 전에 명시적으로 승인해야 합니다.

### 2b단계: MCP 및 A2A 레코드 등록

In [ ]:
# ─── Descriptor builder 구성 ─────────────────────────────────────────────────
# 각 Registry 레코드에는 consumer에게 연결 방법을 알려 주는 구조화된 metadata인
# "descriptors"가 필요합니다. 형식은 descriptor type에 따라 다릅니다.
#
#   MCP 레코드에 필요한 항목:
#     • server  — 서버 이름, version 및 Gateway URL(websiteUrl)
#     • tools   — name, description 및 inputSchema가 있는 도구 목록
#   Orchestrator는 이를 통해 연결할 위치와 사용 가능한 도구를 파악합니다.
#
#   A2A 레코드에 필요한 항목:
#     • agentCard — skill과 기능이 포함된 A2A agent card
#   Orchestrator는 이를 통해 A2A 메시지를 보낼 위치를 파악합니다.

from urllib.parse import quote


def build_mcp_descriptors(name, description, gateway_url, tools):
    """서버 연결 정보와 도구 스키마를 포함한 MCP 디스크립터를 구성합니다.

    참고: 서버 설명은 100자로 제한됩니다.
    전체 설명은 레코드의 최상위 description 필드에 저장되어 시맨틱 검색에
    사용됩니다. 서버 디스크립터에는 잘린 사본이 저장됩니다.
    """
    server_desc = description[:100] if len(description) > 100 else description
    server_content = json.dumps(
        {
            "name": f"gateway-mcp-server/{name}",
            "description": server_desc,
            "version": "1.0.0",
            "websiteUrl": gateway_url,
        }
    )
    tools_content = json.dumps({"tools": tools})
    return {
        "mcp": {
            "server": {"schemaVersion": "2025-12-11", "inlineContent": server_content},
            "tools": {"protocolVersion": "2025-06-18", "inlineContent": tools_content},
        }
    }


def build_a2a_descriptors(name, description, agent_arn):
    """Runtime ARN이 있는 에이전트 카드를 포함한 A2A 디스크립터를 구성합니다.

    에이전트 카드는 Strands A2AServer가 생성하는 A2A 프로토콜 0.3.0 형식을
    따릅니다. URL은 AgentCore 호출 엔드포인트 형식에 맞춰 Runtime ARN으로
    구성합니다.
    """
    escaped_arn = quote(agent_arn, safe="")
    agent_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations/"
    agent_card = {
        "protocolVersion": "0.3.0",
        "name": name.replace("_", " ").title(),
        "description": description,
        "url": agent_url,
        "version": "1.0.0",
        "preferredTransport": "JSONRPC",
        "capabilities": {
            "streaming": False,
            "pushNotifications": False,
        },
        "defaultInputModes": ["text"],
        "defaultOutputModes": ["text"],
        "skills": [
            {
                "id": name,
                "name": name.replace("_", " ").title(),
                "description": description,
                "tags": [],
            }
        ],
    }
    return {
        "a2a": {
            "agentCard": {
                "schemaVersion": "0.3.0",
                "inlineContent": json.dumps(agent_card),
            }
        }
    }


# ─── 각 도구/에이전트를 Registry 레코드로 등록 ──────────────────────────────
# create_registry_record의 파라미터:
#   • name           — 이 Registry 내의 고유 identifier
#   • description    — semantic search를 위해 index에 포함되는 자연어 텍스트
#   • descriptorType — "MCP", "A2A", "CUSTOM" 또는 "AGENT_SKILLS"
#   • descriptors    — 프로토콜별 metadata(위에서 구성)
#   • recordVersion  — 변경 사항 추적용 version 문자열

record_ids = []

for name, cfg in registry_records.items():
    if cfg["protocol"] == "MCP":
        descriptors = build_mcp_descriptors(name, cfg["description"], gateway_url, cfg["tools"])
    else:
        descriptors = build_a2a_descriptors(name, cfg["description"], agent_arns[name])

    resp = cp_client.create_registry_record(
        registryId=REGISTRY_ID,
        name=name,
        description=cfg["description"],
        descriptorType=cfg["protocol"],
        descriptors=descriptors,
        recordVersion="1.0",
    )
    rid = resp["recordArn"].split("/record/")[-1]
    record_ids.append(rid)
    print(f"Created {cfg['protocol']}: {name} -> {rid}")

print(f"\nTotal records: {len(record_ids)}")

# 모든 레코드가 생성되었는지 확인
for rid in record_ids:
    rec = cp_client.get_registry_record(registryId=REGISTRY_ID, recordId=rid)
    desc_type = rec.get("descriptorType", "N/A")
    print(f"  {rec['name']}: status={rec.get('status', 'N/A')}, type={desc_type}")

> ✅ **확인 지점**: MCP 1개(`order_management_mcp`)와 A2A 2개(`pricing_agent`, `customer_support_agent`), 총 3개의 레코드가 생성되고 각 record ID가 출력되어야 합니다.

### 2c단계: 모든 레코드 승인

In [ ]:
# 레코드는 CREATING 상태에서 시작하여 DRAFT로 전환된 후 검색 결과에 표시되기 전에
# 두 단계의 승인을 거쳐야 합니다.
#   DRAFT → PENDING_APPROVAL → APPROVED
#
# 이를 통해 거버넌스 워크플로를 사용할 수 있습니다. 프로덕션 환경에서는
# 담당 검토자 또는 CI pipeline이 승인 단계를 제어할 수 있습니다.

# 모든 레코드가 승인 가능한 상태가 될 때까지 대기
print("Waiting for records to be ready for approval...")
for rid in record_ids:
    while True:
        rec = cp_client.get_registry_record(registryId=REGISTRY_ID, recordId=rid)
        status = rec.get("status", "UNKNOWN")
        if status in ("DRAFT", "PENDING_APPROVAL"):
            break
        print(f"  {rec['name']}: {status} - waiting...")
        time.sleep(5)
    print(f"  {rec['name']}: {status}")

# DRAFT 및 PENDING_APPROVAL 상태를 모두 처리하며 각 레코드의 승인 요청을 제출하고 승인
for rid in record_ids:
    rec = cp_client.get_registry_record(registryId=REGISTRY_ID, recordId=rid)
    status = rec.get("status")

    if status == "DRAFT":
        cp_client.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=rid)

    if status in ("DRAFT", "PENDING_APPROVAL"):
        cp_client.update_registry_record_status(
            registryId=REGISTRY_ID,
            recordId=rid,
            status="APPROVED",
            statusReason="Approved",
        )
    print(f"Approved: {rid}")

# 검색 index는 eventually consistent하므로 모든 레코드가 표시될 때까지 대기
print("\nWaiting for search index to propagate all records...")
for attempt in range(12):
    time.sleep(10)
    resp = dp_client.search_registry_records(
        registryIds=[REGISTRY_ARN],
        searchQuery="order pricing support",
        maxResults=10,
    )
    found = len(resp.get("registryRecords", []))
    if found >= len(record_ids):
        print(f"  All {found} records indexed.")
        break
    print(f"  {found}/{len(record_ids)} records indexed - waiting...")
print("Ready.")

> **문제 해결**: 승인이 `ValidationException`으로 실패하면 제출 전에 각 레코드가 `CREATING`이 아닌 `DRAFT` 상태에 도달했는지 확인하세요. 두 단계가 필요합니다. `submit_registry_record_for_approval`은 레코드를 `PENDING_APPROVAL`로 이동하고, 이어서 `update_registry_record_status(APPROVED)`가 검색 가능한 상태로 만듭니다.

### 2d단계: Semantic Search 확인

In [ ]:
# search_registry_records는 keyword search가 아닌 embedding 기반 유사도 matching을
# 사용합니다. 따라서 이름에 정확히 같은 단어가 없어도 "return refund"가
# "Customer Support Agent"와 일치할 수 있습니다. Search query는 각 레코드의
# name, description 및 tool schema와 비교됩니다.
#
# 서로 다른 기능을 검색할 수 있는지 4개의 query로 테스트합니다.


def get_descriptor_type(record):
    """descriptors 키에서 디스크립터 유형을 판별합니다."""
    descriptors = record.get("descriptors", {})
    if "mcp" in descriptors:
        return "MCP"
    elif "a2a" in descriptors:
        return "A2A"
    return record.get("descriptorType", "UNKNOWN")


for query in [
    "order status tracking",
    "pricing discount promo code",
    "return refund customer support",
    "cancel order update",
]:
    resp = dp_client.search_registry_records(
        registryIds=[REGISTRY_ARN],
        searchQuery=query,
        maxResults=3,
    )
    hits = resp.get("registryRecords", [])
    print(f"\n'{query}' -> {len(hits)} results:")
    for h in hits:
        print(f"  - {h['name']} ({get_descriptor_type(h)})")

> ✅ **확인 지점**: 위의 4개 query가 각각 1~3개의 결과를 반환해야 합니다. 핵심 테스트는 `"return refund customer support"`가 이름에 "return"과 "refund"가 없어도 Customer Support Agent와 일치하는 것입니다. 이것이 semantic search의 작동 방식입니다.

> **문제 해결**: 검색 결과가 0개이면 30초 기다린 후 다시 시도하세요. 레코드 승인 후 검색 index는 **eventually consistent**합니다. 결과가 계속 표시되지 않으면 `cp_client.get_registry_record()`를 사용하여 레코드가 승인되었는지(status = `APPROVED`) 확인하세요.

---

## 3단계: Orchestrator Agent 배포

Orchestrator는 도구를 하드코딩하지 않고 모든 요청마다 AWS Agent Registry에서 실시간으로 검색하는 **agent-within-agent pattern**을 따릅니다.

**Outer Agent**(AgentCore Runtime의 상시 A2A 서버)
- 사용자 요청을 받아 Registry 검색으로 변환
- Cognito로 인증하여 Registry 액세스용 OAuth 자격 증명 획득
- 사용자 요청을 semantic query로 사용하여 **AWS Agent Registry를 실시간 검색**. Registry는 keyword가 아닌 의미를 기반으로 일치하는 MCP 서버와 A2A 에이전트를 반환
- 검색된 레코드를 실행을 위해 inner agent에 전달
- Inner agent가 종합한 응답을 호출자에게 반환

**Inner Agent**(요청마다 Registry 결과로 구성되는 임시 에이전트)
- 특정 요청에 대해 Registry가 일치시킨 도구만 사용하여 동적으로 생성
- Registry가 MCP 서버를 반환하면 `MCPClient`를 사용(Gateway를 통한 OAuth bearer 인증)
- Registry가 A2A 에이전트를 반환하면 `@tool` wrapper를 사용(Runtime을 통한 SigV4 인증)
- LLM이 도구 설명을 읽고 적합한 도구를 선택하여 실행한 후 응답을 종합
- 응답은 outer agent를 거쳐 호출자에게 전달됩니다. 이후 inner agent는 제거되며 다음 요청에서는 새 검색 결과와 도구를 사용합니다.

**중요한 이유:** Orchestrator는 어떤 도구나 에이전트가 존재하는지 전혀 알지 못합니다. 사용하는 모든 기능은 실제 Registry 검색 결과에서 가져옵니다. 새 MCP 서버 또는 A2A 에이전트를 Registry에 등록하면 코드 변경이나 재배포 없이 다음 요청에서 검색하여 사용합니다.

<img src="./images/orchestrator_agent_flow_v3.png" alt="Orchestrator Agent 흐름" width="900"/>

Runtime container에 번들로 포함된 `utils.py`의 helper 함수 `parse_server_metadata`, `create_mcp_client_from_metadata`, `create_a2a_tool_from_metadata`는 Registry metadata를 실제 연결로 변환합니다.

In [ ]:
ORCHESTRATOR_AGENT_CODE = '''
import os
import json
import boto3
from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent.a2a import A2AServer
from fastapi import FastAPI
import uvicorn
from utils import (
    parse_server_metadata,
    create_mcp_client_from_metadata,
    create_a2a_tool_from_metadata,
    fetch_oauth_token,
)

# ─── 환경 변수 구성 ───────────────────────────────────────────────────────────
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
REGISTRY_ARN = os.environ["REGISTRY_ARN"]
COGNITO_DOMAIN = os.environ["COGNITO_DOMAIN"]
CLIENT_ID = os.environ["CLIENT_ID"]
SCOPES = os.environ["SCOPES"]
MODEL_ID = os.environ.get("MODEL_ID", "us.anthropic.claude-sonnet-4-6")

session = boto3.Session()

_sm = session.client("secretsmanager", region_name=REGION)
CLIENT_SECRET = _sm.get_secret_value(
    SecretId=os.environ["CLIENT_SECRET_NAME"]
)["SecretString"]

# bedrock-agentcore DP에는 invoke_agent_runtime과 search_registry_records가 모두 있으며
# Runtime 기본 이미지에 포함된 모든 boto3 버전에서 사용할 수 있음
dp_client = session.client("bedrock-agentcore", region_name=REGION)


@tool
def discover_and_execute(request: str) -> str:
    """Search the AWS Agent Registry for relevant tools and agents, then execute the request
    using the discovered capabilities. Use this for every user request.

    Args:
        request: The user request to process.

    Returns:
        The response from executing the request with dynamically discovered tools.
    """
    access_token = fetch_oauth_token(COGNITO_DOMAIN, CLIENT_ID, CLIENT_SECRET, SCOPES, REGION)

    search_queries = [
        request,
        "order management status tracking cancel update",
        "pricing discount promo code savings",
        "customer support returns refunds complaints",
    ]
    all_records = {}
    for q in search_queries:
        results = dp_client.search_registry_records(
            registryIds=[REGISTRY_ARN], searchQuery=q, maxResults=5,
        ).get("registryRecords", [])
        for rec in results:
            name = rec.get("name", "")
            if name not in all_records:
                all_records[name] = rec

    records = list(all_records.values())

    if not records:
        return "No tools found in registry for this request."

    mcp_clients, a2a_tools, seen_urls = [], [], set()
    for rec in records:
        meta = parse_server_metadata(rec)
        if meta["protocol"] == "MCP":
            url = meta.get("url")
            if url and url not in seen_urls:
                c = create_mcp_client_from_metadata(meta, access_token)
                if c:
                    mcp_clients.append(c)
                    seen_urls.add(url)
        elif meta["protocol"] == "A2A":
            fn = create_a2a_tool_from_metadata(meta, session, REGION)
            if fn:
                a2a_tools.append(fn)

    if not mcp_clients and not a2a_tools:
        return "No tools could be instantiated from registry results."

    model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
    started_clients = []
    try:
        mcp_tools = []
        for c in mcp_clients:
            c.start()
            started_clients.append(c)
            mcp_tools.extend(c.list_tools_sync())

        sub_agent = Agent(
            model=model,
            tools=mcp_tools + a2a_tools,
            system_prompt=(
                "You are an Order Management & Customer Service assistant. "
                "You have access to multiple tools — choose the RIGHT tool for each request:\\n"
                "- For order lookups, status checks, cancellations, or address changes: use the MCP tools (get_order_status, update_order)\\n"
                "- For pricing analysis, discounts, and promo codes: use the pricing_agent tool\\n"
                "- For returns, refunds, complaints, and escalations: use the customer_support_agent tool\\n"
                "Always use the most specific tool for the task. Do NOT use the pricing agent for order status or return questions."
            ),
        )
        return str(sub_agent(request))
    finally:
        for c in started_clients:
            try:
                c.stop()
            except Exception:
                pass


model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
agent = Agent(
    model=model,
    name="Orchestrator Agent",
    description="Order management orchestrator that discovers and invokes tools and agents from the AWS Agent Registry",
    system_prompt=(
        "You are an orchestrator agent. For every user request, use the "
        "discover_and_execute tool to search the registry and process it. "
        "Always pass the full user request to the tool."
    ),
    tools=[discover_and_execute],
)

app = FastAPI()
a2a = A2AServer(
    agent=agent,
    http_url=os.environ.get("AGENTCORE_RUNTIME_URL", "http://127.0.0.1:9000/"),
    serve_at_root=True,
)

@app.get("/ping")
def ping():
    return {"status": "healthy"}

app.mount("/", a2a.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=9000)
'''

# Orchestrator 에이전트를 disk에 작성
with open("orchestrator_agent.py", "w") as f:
    f.write(ORCHESTRATOR_AGENT_CODE)

# Orchestrator 종속성
with open("orchestrator_requirements.txt", "w") as f:
    f.write("strands-agents[a2a]\nfastapi\nuvicorn\nmcp\nrequests\n")

print("✓ Orchestrator agent code written: orchestrator_agent.py")

# 새로 생성하도록 오래된 Dockerfile 삭제
if os.path.exists("Dockerfile"):
    os.remove("Dockerfile")
    print("✓ Deleted stale Dockerfile (will be regenerated)")

# Orchestrator를 Runtime에 배포
orchestrator_rt = Runtime()
orchestrator_rt.configure(
    entrypoint="orchestrator_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="orchestrator_requirements.txt",
    region=region,
    agent_name="orchestrator_agent",
    protocol="A2A",
)
orchestrator_launch = orchestrator_rt.launch(
    auto_update_on_conflict=True,
    env_vars={
        "REGISTRY_ARN": REGISTRY_ARN,
        "COGNITO_DOMAIN": cognito_domain,
        "CLIENT_ID": client_id,
        "CLIENT_SECRET_NAME": secret_name,
        "SCOPES": scopes,
        "MODEL_ID": MODEL_ID,
        "AWS_DEFAULT_REGION": region,
    },
)
orchestrator_agent_id = orchestrator_launch.agent_id
orchestrator_arn = orchestrator_launch.agent_arn
print(f"✓ Orchestrator deployed: {orchestrator_agent_id}")

# Orchestrator의 execution role에 필요한 권한 부여
agent_info = cp_client.get_agent_runtime(agentRuntimeId=orchestrator_agent_id)
orch_role_arn = agent_info.get("roleArn") or agent_info.get("agentRuntimeRoleArn", "")
if orch_role_arn:
    orch_role_name = orch_role_arn.split("/")[-1]

    iam_client.put_role_policy(
        RoleName=orch_role_name,
        PolicyName="SecretsManagerReadAccess",
        PolicyDocument=json.dumps(
            {
                "Version": "2012-10-17",
                "Statement": [
                    {
                        "Effect": "Allow",
                        "Action": "secretsmanager:GetSecretValue",
                        "Resource": f"arn:aws:secretsmanager:{region}:{account_id}:secret:{secret_name}*",
                    }
                ],
            }
        ),
    )
    print(f"✓ Secrets Manager access granted to: {orch_role_name}")

    iam_client.put_role_policy(
        RoleName=orch_role_name,
        PolicyName="RegistrySearchAccess",
        PolicyDocument=json.dumps(
            {
                "Version": "2012-10-17",
                "Statement": [
                    {
                        "Effect": "Allow",
                        "Action": "bedrock-agentcore:SearchRegistryRecords",
                        "Resource": f"arn:aws:bedrock-agentcore:*:{account_id}:registry/*",
                    }
                ],
            }
        ),
    )
    print(f"✓ Registry search access granted to: {orch_role_name}")

print(f"✓ Orchestrator status: {agent_info['status']}, arn: {orchestrator_arn}")

---

## 4단계: End-to-End 데모 - 3가지 시나리오

각 데모는 서로 다른 기능 조합으로 전체 검색 → 인스턴스화 → 실행 주기를 보여 줍니다. 아래 다이어그램은 데모 3(반품 및 환불)의 네 단계를 보여 줍니다. **(1)** Orchestrator가 "order return refund"로 Registry 검색, **(2)** 일치하는 MCP 서버를 호출하여 주문 세부 정보 조회, **(3)** 일치하는 A2A 에이전트를 호출하여 policy 평가, **(4)** 고객 응답 종합.

### 데모 1: 주문 상태 - MCP 도구 호출

In [ ]:
from utils import invoke_orchestrator

# 데모 1: 주문 상태
result = invoke_orchestrator(
    "What is the current status and tracking info for order 123?",
    agentcore_client,
    orchestrator_arn,
)
print(f"\n── Response ──\n{result}")

#### 데모 1 실행 내용

1. **검색**: Registry semantic search가 "order status"와 일치하는 주문 관리 MCP 서버를 반환했습니다.
2. **인스턴스화**: Orchestrator가 Gateway MCP 서버용 `MCPClient`를 생성했습니다.
3. **실행**: LLM이 orderId "123"으로 `get_order_status`를 호출하고 읽기 쉬운 응답을 종합했습니다.

### 데모 2: 가격 및 할인 - MCP + A2A Multi-Agent 협업

In [ ]:
result = invoke_orchestrator(
    "Order 123 has 2x Widget Pro at $99.98. What discount tiers or promo codes can reduce the price?",
    agentcore_client,
    orchestrator_arn,
)
print(f"\n── Response ──\n{result}")

#### 데모 2 실행 내용

이 데모는 하나의 요청에 대해 **Registry가 여러 기능을 검색**하는 방식을 보여 줍니다.

1. **검색**: 하나의 자연어 query로 Registry가 주문 관리 MCP 서버와 가격 책정 A2A 에이전트라는 서로 다른 두 프로토콜을 모두 일치시켰습니다.
2. **MCP 서버**: Gateway를 통해 `get_order_status`를 호출하여 주문 세부 정보를 가져왔습니다.
3. **A2A 에이전트**: 할인 분석을 위해 주문 데이터를 Runtime의 가격 책정 에이전트에 전송했습니다.
4. **종합**: LLM이 두 응답을 결합하여 일관된 가격 추천을 만들었습니다.

### 데모 3: 반품 및 환불 - 고객 지원 판단

In [ ]:
result = invoke_orchestrator(
    "I want to return order 789 (Premium Headphones, delivered March 10). Am I eligible for a return and what is the refund amount?",
    agentcore_client,
    orchestrator_arn,
)
print(f"\n── Response ──\n{result}")

#### 데모 3 실행 내용

위의 흐름 다이어그램은 이 시나리오를 그대로 보여 줍니다. Registry의 핵심은 등록된 이름에 정확히 같은 단어가 없어도 "return refund" query가 고객 지원 A2A 에이전트와 **의미상 일치**했다는 점입니다. Registry가 없다면 orchestrator에 반품을 처리하는 에이전트 정보를 하드코딩해야 합니다.

---

## 정리

모든 리소스를 역순으로 삭제합니다.

In [ ]:
# 위에서 생성한 모든 AWS 리소스를 역순으로 삭제하는 정리 script를 실행합니다.
# 이 script는 Notebook kernel의 변수(gateway_id, lambda_arns 등)를 사용합니다.
# 자세한 내용은 cleanup.py를 참조하세요.
%run -i cleanup.py

# 축하합니다!

**AWS Agent Registry**를 사용하여 다음 작업을 수행하는 자율 에이전트를 성공적으로 구축했습니다.

1. 하드코딩된 통합 없이 semantic search를 통해 runtime에 MCP 서버와 에이전트 **검색**
2. Registry metadata를 사용하여 MCP 서버(Amazon Bedrock AgentCore Gateway 사용) 및 A2A 에이전트(Amazon Bedrock AgentCore Runtime 사용)에 동적으로 **연결**
3. Orchestrator가 각 요청에 적합한 기능을 찾는 multi-agent 워크플로 **실행**

### 핵심 요점
- **AWS Agent Registry가 핵심 역할 수행**: 정적이고 하드코딩된 에이전트 시스템을 동적 시스템으로 전환합니다. 한 번 등록하면 어디서든 검색할 수 있습니다.
- **Keyword matching이 아닌 semantic search**: Orchestrator가 필요한 기능을 자연어로 설명하면 Registry가 여러 프로토콜에서 가장 적합한 결과를 반환합니다.
- **프로토콜에 구애받지 않는 검색**: 하나의 Registry 검색이 MCP 서버와 A2A 에이전트를 모두 반환합니다. Orchestrator는 연결할 때까지 기능이 사용하는 프로토콜을 알 필요가 없습니다.
- **재배포 불필요**: 새 MCP 서버 또는 A2A 에이전트를 Registry에 추가하면 orchestrator가 다음 요청에서 검색합니다.

### 지원 인프라
- **Amazon Bedrock AgentCore Gateway** — AWS Lambda 함수를 인증 기능이 내장된 관리형 MCP 서버로 전환
- **Amazon Bedrock AgentCore Runtime** — IAM 기반 보안으로 A2A 에이전트 호스팅

### 구현한 사용 사례
- **Order Management MCP** — Amazon Bedrock AgentCore Gateway를 통한 stateless 주문 조회 및 업데이트
- **Pricing Agent (A2A)** — 할인, promo code 및 가격 기록 평가
- **Customer Support Agent (A2A)** — 반품 가능 여부 평가, 환불 계산 및 policy 규칙 적용

## 다음 단계

- [AgentCore samples repo](https://github.com/awslabs/agentcore-samples)에서 더 많은 튜토리얼 살펴보기
- [Amazon Bedrock AgentCore 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/userguide/) 읽기
- **이 pattern 확장**: 새 MCP 서버 또는 A2A 에이전트를 Registry에 추가하면 orchestrator가 자동으로 검색합니다.
- **모델 교체**: 다른 foundation model을 사용하려면 `MODEL_ID`를 변경합니다.